In [25]:
%reload_ext autoreload
%autoreload 2

############################################################################
### This notebook is for computing fixation probabilities in the context ###
### of selection only (no mutations) in finite populations               ###

##############################################
###   EXPLICIT WRIGHT-FISHER SIMULATIONS   ###
##############################################

### This version does a scan over D equally spaced values of s_alpha, alpha = 1 .. M-1 ###

from functions_WF import *
import time

start_time = time.time()

folder = "data_fix_prob"
#folder = "./"

Nruns = 20000 # total number of runs for computing fixation probabilities for a given combination of fitnesses
#Nruns = 50000 # total number of runs for computing fixation probabilities for a given combination of fitnesses
Ltot = 1000000 # max number of steps (should never be reached due to fixation)

N = 100 # total number of individuals in the population
M = 3 # total number of genotypes
D = 10 # total number of points per fitness range
#M = 3 # total number of genotypes
#D = 2 # total number of points per fitness range

READN0 = False

##################################

rng = np.random.default_rng()

# Initial population counts and frequencies:
assert M > 1 and type(M) is int, "Illegal M!"

file_n0 = os.path.join(folder, f"fix_probs_WF.Nruns{Nruns}_N{N}_M{M}_D{D}.n0.txt")
####
if READN0 == True:
    init_counts = np.loadtxt(file_n0, dtype=int)
    assert init_counts.size == M, "Dimension mismatch in initial counts!"
else:
    init_counts = np.array(balanced_partition(N, M), dtype=int) # as uniform as possible init_counts
    np.savetxt(file_n0, init_counts, fmt='%d')

init_freqs = init_counts / N 


print(f"M = {M}")
np.set_printoptions(formatter={'float': '{:0.3f}'.format})
print("Initial population frequencies:\n",init_freqs)
np.set_printoptions()

n_init = []
for j in range(M):
    n_init.extend([j] * init_counts[j])
n_init = np.array(n_init)
                           
### Set up selection coeffs: ###
eps = 1e-05 # s = 0.0 does not work in Kimura's formula
SMAX = 0.02
sel_coeff = np.linspace(0.0, SMAX, num=D) + eps

sel_shape = tuple(np.repeat(D,M-1)) # shape of the matrix of selection coeff. 

fix_probs = np.zeros(sel_shape + (M,)) # fixation probs
fix_counts = np.zeros(sel_shape + (M,), dtype=int) # fixation counts

#debug
#print("fix_counts shape =",fix_counts.shape)

print(f"Starting {D**(M-1)} WF simulations ..")
Mstep = int(D**(M-1) // 10) # intervals at which progress is reported

######
for k in range(D**(M-1)): # "global" index over selection coeffs
    
    ####
    if (Mstep > 0 and (k+1)%Mstep == 0) or (Mstep == 0):
        if k+1 == 1:
            print("{} run ..".format(k+1))
        else:
            print("{} runs ..".format(k+1))
            
    
    crd_cur = tuple(np.unravel_index(k, sel_shape))
    assert len(crd_cur) == M-1, "Dimension mismatch!"
    
    #debug
    #print(f"crd_cur = {crd_cur}")
    
    sel_vec = np.zeros(M)
    #sel_vec[-1] = 0.0 # last fitness value is always 1.0 by convention
    
    ### Selection coefficients for each genotype: ###
    for p in range(M-1):
        sel_vec[p] = sel_coeff[crd_cur[p]]
    
    F_vals = sel_vec + 1.0 # array of fitness values, length M
    
    ####################
    # Build fixation probs given a vector of selection coeffs and initial population freq:
    for i in range(Nruns):
    
        #print(f"\n** run = {i+1} **")

        # Initial population:
        n = n_init.copy()
        #x = get_freqs(n, M)

        assert len(n) == N, "Dimension mismatch in the counts array!"
        
        F_arr = F_vals[n]
        #F_arr = np.zeros(N) # initial fitness values

        #debug
        #print("Initial population:\n",n)
        #np.set_printoptions(formatter={'float': '{:0.3f}'.format})
        #print("Initial freqs:\n",x)
        #np.set_printoptions()
        #print("Initial <F> = {:.4f}".format(np.mean(F_arr)))
        #print("Initial std(<F>) = {:.4f}".format(np.std(F_arr)))
        
        
        # Evolve the population to fixation:
        flag_fixed = False
        for l in range(Ltot):

            #Fnorm = F_arr/np.sum(F_arr)
            Fnorm = F_arr / F_arr.sum(dtype=float)

            #debug
            #print(Fnorm)

            #new_ind = np.sort(np.random.choice(N, size=N, replace=True, p=Fnorm))
            #new_ind = np.sort(rng.choice(N, size=N, replace=True, p=Fnorm))
            new_ind = rng.choice(N, size=N, replace=True, p=Fnorm) # sort removed
            n = n[new_ind] # updated genotype indices
            cnt_n = get_counts(n, M)
            
            F_arr = F_vals[n] # updated fitness values
            
            #debug
            #print("new_ind =",new_ind)

            #if (cnt_n == N).any() == True: # fixation!
            if np.max(cnt_n) == N: # fixation!
                indices = np.where(cnt_n == N)[0].tolist()
                assert len(indices) == 1, "Fixation of multiple alleles?!"
                ######
                fix_counts[crd_cur + (indices[0],)] += 1
                #debug
                #print(crd_cur + (indices[0],))
                
                flag_fixed = True
                break
           
        if flag_fixed == False:
            print("WARNING: Population failed to fix!")
            
    # Compute fixation probs:
    #debug
    #print("fix_counts =",np.sum(fix_counts[crd_cur]))
    
    count_sum = np.sum(fix_counts[crd_cur])
    assert count_sum == Nruns, f"Normalization mismatch: {count_sum} != {Nruns}"
    
    fix_probs[crd_cur] = fix_counts[crd_cur]/Nruns


#np.set_printoptions(formatter={'float': '{:0.3f}'.format})
##np.set_printoptions(precision=3, suppress=True)
#print("Fitness values by genotype:\n",F_vals)
#print("==================")
#np.set_printoptions()
################################################


filename = os.path.join(folder, f"fix_probs_WF.Nruns{Nruns}_N{N}_M{M}_D{D}.npy")
np.save(filename, fix_probs)

print(f"Finished {D**(M-1)} WF simulations ..")

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")


M = 3
Initial population frequencies:
 [0.330 0.330 0.340]
Starting 100 WF simulations ..
10 runs ..
20 runs ..
30 runs ..
40 runs ..
50 runs ..
60 runs ..
70 runs ..
80 runs ..
90 runs ..
100 runs ..
Finished 100 WF simulations ..

Code ran in 15640.8510 seconds = 260.6808 minutes


In [28]:
%reload_ext autoreload
%autoreload 2

##############################################
###   EXPLICIT WRIGHT-FISHER SIMULATIONS   ###
##############################################

### This version uses D randomly chosen [s_1 .. s_M-1], NO scan is performed ###

from functions_WF import *
import time

start_time = time.time()

folder = "data_fix_prob"
#folder = "./"

Nruns = 20000 # total number of runs for computing fixation probabilities for a given combination of fitnesses
#Nruns = 50000 # total number of runs for computing fixation probabilities for a given combination of fitnesses
Ltot = 1000000 # max number of steps (should never be reached due to fixation)

N = 100 # total number of individuals in the population
##M = 3 # total number of genotypes
#M = 5 # total number of genotypes
#D = 20 # total number of randomly sampled selection coefficient sets
#M = 10 # total number of genotypes
#D = 10 # total number of randomly sampled selection coefficient sets
M = 20 # total number of genotypes
D = 5 # total number of randomly sampled selection coefficient sets

READN0 = False # reads initial counts from a file if True
READSEL = False # reads selection coefficients from a file if True

##################################

rng = np.random.default_rng()

# Initial population counts and frequencies:
assert M > 1 and type(M) is int, "Illegal M!"

file_n0 = os.path.join(folder, f"fix_probs_WF_rsets.Nruns{Nruns}_N{N}_M{M}_D{D}.n0.txt")
####
if READN0 == True:
    init_counts = np.loadtxt(file_n0, dtype=int)
    assert init_counts.size == M, "Dimension mismatch in initial counts!"
else:
    init_counts = np.array(balanced_partition(N, M), dtype=int) # as uniform as possible init_counts
    np.savetxt(file_n0, init_counts, fmt='%d')

init_freqs = init_counts / N


print(f"M = {M}")
np.set_printoptions(formatter={'float': '{:0.3f}'.format})
print("Initial population frequencies:\n",init_freqs)
np.set_printoptions()

n_init = []
for j in range(M):
    n_init.extend([j] * init_counts[j])
n_init = np.array(n_init)
                           

### Set up selection coeffs: ###
filename1 = os.path.join(folder, f"sel_coeff_WF_rsets.N{N}_M{M}_D{D}.npy")
####
if READSEL == True:
    sel_coeff_set = np.load(filename1)
    assert sel_coeff_set.shape[0] == D and sel_coeff_set.shape[1] == M-1, "Dimension mismatch in sel_coeff_set!"
else:
    eps = 1e-05 # s = 0.0 does not work in Kimura's formula
    SMAX = 0.02
    sel_coeff_set = rng.uniform(0.0, SMAX, size=(D,M-1)) + eps
    np.save(filename1, sel_coeff_set)
    
zeros_col = np.zeros((sel_coeff_set.shape[0], 1))

# Stack horizontally
sel_coeff = np.hstack((sel_coeff_set, zeros_col))

######
fix_probs = np.zeros((D,M)) # global fixation probs
fix_counts = np.zeros((D,M)) # global fixation counts

print(f"Starting {D} WF simulation(s) ..")
Mstep = int(D // 5) # intervals at which progress is reported

for k in range(D): # "global" index over selection coeffs
    
    ####
    if (Mstep > 0 and (k+1)%Mstep == 0) or (Mstep == 0):
        if k+1 == 1:
            print("{} run ..".format(k+1))
        else:
            print("{} runs ..".format(k+1))
    
    sel_vec = sel_coeff[k]
    
    
    F_vals = sel_vec + 1.0 # array of fitness values, length M
    
    ####################
    # Build fixation probs given a vector of selection coeffs and initial population freq:
    for i in range(Nruns):
    
        #print(f"\n** run = {i+1} **")

        # Initial population:
        n = n_init.copy()
        #x = get_freqs(n, M)
        
        assert len(n) == N, "Dimension mismatch in the counts array!"
        
        F_arr = F_vals[n]
        #F_arr = np.zeros(N) # initial fitness values

        #debug
        #print("Initial population:\n",n)
        #np.set_printoptions(formatter={'float': '{:0.3f}'.format})
        #print("Initial freqs:\n",x)
        #np.set_printoptions()
        #print("Initial <F> = {:.4f}".format(np.mean(F_arr)))
        #print("Initial std(<F>) = {:.4f}".format(np.std(F_arr)))
        
        
        # Evolve the population to fixation:
        flag_fixed = False
        for l in range(Ltot):

            Fnorm = F_arr / F_arr.sum(dtype=float)
            
            new_ind = rng.choice(N, size=N, replace=True, p=Fnorm) # sort removed
            n = n[new_ind] # updated genotype indices
            cnt_n = get_counts(n, M)
            
            F_arr = F_vals[n] # updated fitness values

            #if (cnt_n == N).any() == True: # fixation!
            if np.max(cnt_n) == N: # fixation!
                indices = np.where(cnt_n == N)[0].tolist()
                assert len(indices) == 1, "Fixation of multiple alleles?!"
                ######
                fix_counts[k,indices[0]] += 1
                
                flag_fixed = True
                break
           
        if flag_fixed == False:
            print("WARNING: Population failed to fix!")
            
    # Compute fixation probs:
    count_sum = np.sum(fix_counts[k])
    assert count_sum == Nruns, f"Normalization mismatch: {count_sum} != {Nruns}"
    
    fix_probs[k] = fix_counts[k]/Nruns


#filename1 = f"sel_coeff_WF_rsets.N{N}_M{M}_D{D}.npy"
#np.save(filename1, sel_coeff_set)

filename2 = os.path.join(folder, f"fix_probs_WF_rsets.Nruns{Nruns}_N{N}_M{M}_D{D}.npy")
np.save(filename2, fix_probs)

print(f"Finished {D} WF simulation(s) ..")

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")


M = 20
Initial population frequencies:
 [0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050
 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050]
Starting 5 WF simulation(s) ..
1 run ..
2 runs ..
3 runs ..
4 runs ..
5 runs ..
Finished 5 WF simulation(s) ..

Code ran in 872.7162 seconds = 14.5453 minutes


In [29]:
%reload_ext autoreload
%autoreload 2

###                        FAST VERSION (uses numba)                      ###
### This notebook is for checking the fluctuation theorems in the context ###
### of selection only (no mutations) in finite populations                ###
### Langevin simulations for gene frequencies (Euler–Maruyama scheme)     ###

### This version does a scan over D equally spaced values of s_alpha, alpha = 1 .. M-1 ###

from functions_Langevin import *
import time

start_time = time.time()

folder = "data_fix_prob"
#folder = "./"

Nruns = 20000 # total number of runs for computing fixation probabilities for a given combination of fitnesses
Ltot = 10000000 # max number of steps (should never be reached due to fixation)

N = 100 # total number of individuals in the population
M = 3 # total number of genotypes

D = 10 # total number of points per fitness range

#dt = 0.01 # time step in the Langevin simulation
# Regular tolerance:
#dt = 1.0/(5*N) # N-dependent time step in the Langevin simulation
#tol = 1/(2*N) # fixation tolerance, must be larger than dt
# Improved tolerance:
dt = 1.0/(20*N) # N-dependent time step in the Langevin simulation
tol = 1/(4*N) # fixation tolerance, must be larger than dt

READX0 = False # reads initial freqs from a file if True

##################################

# Initial population frequencies:
assert M > 1 and type(M) is int, "Illegal M!"

file_x0 = os.path.join(folder, f"fix_probs_Langevin.Nruns{Nruns}_N{N}_M{M}_D{D}.x0.txt")
####
if READX0 == True:
    init_freqs = np.loadtxt(file_x0)
    assert init_freqs.size == M, "Dimension mismatch in initial freqs!"
else:
    init_freqs = np.ones(M) / M # uniform init_freqs
    np.savetxt(file_x0, init_freqs, fmt='%.8f')


print(f"M = {M}")
np.set_printoptions(formatter={'float': '{:0.3f}'.format})
print("Initial population frequencies:\n",init_freqs)
np.set_printoptions()


### Set up selection coeffs: ###
#eps = 1e-03 # s = 0.0 does not work in Kimura's formula
#sel_coeff = np.linspace(0.0, 0.03, num=D) + eps
eps = 1e-05 # s = 0.0 does not work in Kimura's formula
SMAX = 0.02
sel_coeff = np.linspace(0.0, SMAX, num=D) + eps

sel_shape = tuple(np.repeat(D,M-1)) # shape of the matrix of selection coeff. 

######
fix_probs = np.zeros(sel_shape + (M,)) # global fixation probs

print(f"Starting {D**(M-1)} Langevin simulation(s) ..")
Mstep = int(D**(M-1) // 10) # intervals at which progress is reported

######
sel_vec = np.zeros(M-1)

for k in range(D**(M-1)): # "global" index over selection coeffs
    
    ####
    if (Mstep > 0 and (k+1)%Mstep == 0) or (Mstep == 0):
        if k+1 == 1:
            print("{} run ..".format(k+1))
        else:
            print("{} runs ..".format(k+1))
            
    
    crd_cur = tuple(np.unravel_index(k, sel_shape))
    assert len(crd_cur) == M-1, "Dimension mismatch!"
    
    #debug
    #print(f"crd_cur = {crd_cur}")
    
    ### Selection coefficients for each genotype: ###
    for p in range(M-1):
        sel_vec[p] = sel_coeff[crd_cur[p]]
    
    #debug
    #print("sel_vec =",sel_vec)
    
    ####################
    # Build fixation probs given a vector of selection coeffs and initial population freq:
    counts = run_fixation(init_freqs, sel_vec, Nruns, dt, N, Ltot, tol)
    
    count_sum = np.sum(counts)
    #assert count_sum == Nruns, f"Normalization mismatch: {count_sum} != {Nruns}"
    if count_sum != Nruns:
        print(f"WARNING: normalization mismatch => {count_sum} != {Nruns}")
    
    #fix_probs[crd_cur] = counts/Nruns
    fix_probs[crd_cur] = counts/count_sum


filename = os.path.join(folder, f"fix_probs_Langevin.Nruns{Nruns}_N{N}_M{M}_D{D}.npy")
np.save(filename, fix_probs)

print(f"Finished {D**(M-1)} Langevin simulation(s) ..")

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")


M = 3
Initial population frequencies:
 [0.333 0.333 0.333]
Starting 100 Langevin simulation(s) ..
10 runs ..
20 runs ..
30 runs ..
40 runs ..
50 runs ..
60 runs ..
70 runs ..
80 runs ..
90 runs ..
100 runs ..
Finished 100 Langevin simulation(s) ..

Code ran in 79528.8396 seconds = 1325.4807 minutes


In [32]:
%reload_ext autoreload
%autoreload 2

###                        FAST VERSION (uses numba)                      ###
### This notebook is for checking the fluctuation theorems in the context ###
### of selection only (no mutations) in finite populations                ###
### Langevin simulations for gene frequencies (Euler–Maruyama scheme)     ###

### This version uses D randomly chosen [s_1 .. s_M-1], NO scan is performed ###

from functions_Langevin import *
import time

start_time = time.time()

folder = "data_fix_prob"
#folder = "./"

Nruns = 20000 # total number of runs for computing fixation probabilities for a given combination of fitnesses
Ltot = 10000000 # max number of steps (should never be reached due to fixation)

N = 100 # total number of individuals in the population

##M = 3 # total number of genotypes
#M = 5 # total number of genotypes
#D = 20 # total number of randomly sampled selection coefficient sets
#M = 10 # total number of genotypes
#D = 10 # total number of randomly sampled selection coefficient sets
M = 20 # total number of genotypes
D = 5 # total number of randomly sampled selection coefficient sets

#dt = 0.01 # time step in the Langevin simulation
# Regular tolerance:
#dt = 1.0/(5*N) # N-dependent time step in the Langevin simulation
#tol = 1/(2*N) # fixation tolerance, must be larger than dt
# Improved tolerance:
dt = 1.0/(20*N) # N-dependent time step in the Langevin simulation
tol = 1/(4*N) # fixation tolerance, must be larger than dt

READX0 = False # reads initial freqs from a file if True
READSEL = True # reads selection coefficients from a file if True

##################################

# Initial population frequencies:
assert M > 1 and type(M) is int, "Illegal M!"

file_x0 = os.path.join(folder, f"fix_probs_Langevin_rsets.Nruns{Nruns}_N{N}_M{M}_D{D}.x0.txt")
####
if READX0 == True:
    init_freqs = np.loadtxt(file_x0)
    assert init_freqs.size == M, "Dimension mismatch in initial freqs!"
else:
    init_freqs = np.ones(M) / M # uniform init_freqs
    np.savetxt(file_x0, init_freqs, fmt='%.8f')


print(f"M = {M}")
np.set_printoptions(formatter={'float': '{:0.3f}'.format})
print("Initial population frequencies:\n",init_freqs)
np.set_printoptions()

# Create a generator instance
rng = np.random.default_rng()

### Set up selection coeffs: ###
filename1 = os.path.join(folder, f"sel_coeff_Langevin_rsets.N{N}_M{M}_D{D}.npy")
####
if READSEL == True:
    sel_coeff_set = np.load(filename1)
    assert sel_coeff_set.shape[0] == D and sel_coeff_set.shape[1] == M-1, "Dimension mismatch in sel_coeff_set!"
else:
    eps = 1e-05 # s = 0.0 does not work in Kimura's formula
    SMAX = 0.02
    sel_coeff_set = rng.uniform(0.0, SMAX, size=(D,M-1)) + eps
    np.save(filename1, sel_coeff_set)

######
fix_probs = np.zeros((D,M)) # global fixation probs

print(f"Starting {D} Langevin simulation(s) ..")
Mstep = int(D // 5) # intervals at which progress is reported

######
#sel_vec = np.zeros(M-1)

for k in range(D): # "global" index over selection coeffs
    
    ####
    if (Mstep > 0 and (k+1)%Mstep == 0) or (Mstep == 0):
        if k+1 == 1:
            print("{} run ..".format(k+1))
        else:
            print("{} runs ..".format(k+1))
    
    ####################
    # Build fixation probs given a vector of selection coeffs and initial population freq:
    counts = run_fixation(init_freqs, sel_coeff_set[k], Nruns, dt, N, Ltot, tol)
    
    count_sum = np.sum(counts)
    #assert count_sum == Nruns, f"Normalization mismatch: {count_sum} != {Nruns}"
    if count_sum != Nruns:
        print(f"WARNING: normalization mismatch => {count_sum} != {Nruns}")
    
    fix_probs[k] = counts/count_sum

    
#filename1 = f"sel_coeff_Langevin_rsets.N{N}_M{M}_D{D}.npy"
#np.save(filename1, sel_coeff_set)

filename2 = os.path.join(folder, f"fix_probs_Langevin_rsets.Nruns{Nruns}_N{N}_M{M}_D{D}.npy")
np.save(filename2, fix_probs)

print(f"Finished {D} Langevin simulation(s) ..")

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")


M = 20
Initial population frequencies:
 [0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050
 0.050 0.050 0.050 0.050 0.050 0.050 0.050 0.050]
Starting 5 Langevin simulation(s) ..
1 run ..
2 runs ..
3 runs ..
4 runs ..
5 runs ..
Finished 5 Langevin simulation(s) ..

Code ran in 20509.7545 seconds = 341.8292 minutes


In [1]:
%reload_ext autoreload
%autoreload 2

##############################################
# Load previously computed symbolic data ONCE
##############################################

from functions_expansion import *
import time

start_time = time.time()

##########################

load_symbolic_caches(filename="symbolic_partition_caches_maxk36.pkl") # big library (kmax=36)
#load_symbolic_caches(filename="symbolic_partition_caches_maxk32.pkl") # smaller library (kmax=32) for faster loading

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")


Loaded symbolic caches: 99096 merges, 81156 selections, 49735 Y expansions, 37 Q_k objects, 36 flattened Q_k.

Code ran in 591.4955 seconds = 9.8583 minutes


In [2]:
%reload_ext autoreload
%autoreload 2

from functions_expansion import *
import time

start_time = time.time()

##########################

M = 2
N = 100

gamma = 0 # P_{fix} index

#x = np.ones(M) / M # uniform init_freqs
x = np.array([0.5, 0.5])

s = np.array([-0.020, 0.0])

# Load previously computed symbolic data (if available)
#load_symbolic_caches()

# Neg_tol = make max_k iterations, no exit on tol
P_exp = fixation_probability(x=x, s=s, gamma=gamma, N=N, tol=-1e-14, max_k=36, return_details=True)
print("\nExpansion prob =",P_exp["P_fix"])

print_fixation_details(P_exp, digits=6)

P_Kimura = (1 - np.exp(-2*N*s[0]*x[0]))/(1 - np.exp(-2*N*s[0]))
print("Kimura prob =",P_Kimura,"\n")

# Save any newly generated symbolic data
save_symbolic_caches() # save any changes with default filename

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")



Expansion prob = 0.11920294712250033

P_fix       = 1.192029e-01
Series sum  = 2.384059e-01
Orders used = 36

   k            Q_k           N^k Q_k  Contribution to P_fix
------------------------------------------------------------
   1  -1.000000e-02     -1.000000e+00       -5.000000e-01
   2   0.000000e+00      0.000000e+00        0.000000e+00
   3   3.333333e-07      3.333333e-01        1.666667e-01
   4  -6.203855e-25     -6.203855e-17       -3.101927e-17
   5  -1.333333e-11     -1.333333e-01       -6.666667e-02
   6  -2.524355e-29     -2.524355e-17       -1.262177e-17
   7   5.396825e-16      5.396825e-02        2.698413e-02
   8  -2.311116e-33     -2.311116e-17       -1.155558e-17
   9  -2.186949e-20     -2.186949e-02       -1.093474e-02
  10  -2.350989e-38     -2.350989e-18       -1.175494e-18
  11   8.863236e-25      8.863236e-03        4.431618e-03
  12   2.152394e-42      2.152394e-18        1.076197e-18
  13  -3.592128e-29     -3.592128e-03       -1.796064e-03
  14   1.7516

In [3]:
%reload_ext autoreload
%autoreload 2

from functions_expansion import *
import time

start_time = time.time()

##########################

M = 3
N = 100

gamma = 0 # P_{fix} index

#x = np.ones(M) / M # uniform init_freqs
x = [0.3333, 0.3333, 0.3334]

#s = np.array([0.025, 0.0002, 0.0])
s = np.array([0.01, -0.01, 0.0]) # interference between beneficial and deleterious mutations

# Load previously computed symbolic data (if available)
#load_symbolic_caches()

# Neg_tol = make max_k iterations, no exit on tol
P_ds = fixation_probability(x=x, s=s, gamma=gamma, N=N, tol=-1e-05, max_k=36, return_details=True)
print("\nExpansion prob =",P_ds["P_fix"])

print_fixation_details(P_ds, digits=6)

# Save any newly generated symbolic data
save_symbolic_caches() # save any changes with default filename

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")



Expansion prob = 0.6434650659730423

P_fix       = 6.434651e-01
Series sum  = 1.930588e+00
Orders used = 36

   k            Q_k           N^k Q_k  Contribution to P_fix
------------------------------------------------------------
   1   1.000000e-02      1.000000e+00        3.333000e-01
   2   1.111333e-05      1.111333e-01        3.704074e-02
   3  -2.222000e-07     -2.222000e-01       -7.405926e-02
   4  -1.728642e-10     -1.728642e-02       -5.761564e-03
   5   7.900247e-12      7.900247e-02        2.633152e-02
   6   2.351817e-15      2.351817e-03        7.838605e-04
   7  -3.103652e-16     -3.103652e-02       -1.034447e-02
   8  -2.987618e-20     -2.987618e-04       -9.957730e-05
   9   1.248426e-20      1.248426e-02        4.161004e-03
  10   3.638902e-25      3.638902e-05        1.212846e-05
  11  -5.050337e-25     -5.050337e-03       -1.683277e-03
  12  -4.306680e-30     -4.306680e-06       -1.435417e-06
  13   2.045887e-29      2.045887e-03        6.818942e-04
  14   4.99143

In [22]:
%reload_ext autoreload
%autoreload 2

from functions_expansion import *
import time

start_time = time.time()

##########################

M = 3000
N = 100

gamma = 0 # P_{fix} index

x = np.ones(M) / M # uniform init_freqs

# Initialize the modern generator
rng = np.random.default_rng()

s = np.array(rng.uniform(0.0, 0.002, size=M))
s[0] = 0.02
s[-1] = 0.0

# Load previously computed symbolic data (if available)
#load_symbolic_caches()

# Neg_tol = make max_k iterations, no exit on tol
P_ds = fixation_probability(x=x, s=s, gamma=gamma, N=N, tol=-1e-05, max_k=36, return_details=True)
print("\nExpansion prob =",P_ds["P_fix"])

print_fixation_details(P_ds, digits=6)

# Save any newly generated symbolic data
save_symbolic_caches() # save any changes with default filename

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")



Expansion prob = 0.0012939660063383779

P_fix       = 1.293966e-03
Series sum  = 3.881898e+00
Orders used = 36

   k            Q_k           N^k Q_k  Contribution to P_fix
------------------------------------------------------------
   1   1.899284e-02      1.899284e+00        6.330946e-04
   2   1.200883e-04      1.200883e+00        4.002943e-04
   3  -2.931427e-09     -2.931427e-03       -9.771425e-07
   4  -2.900988e-09     -2.900988e-01       -9.669959e-05
   5   1.752899e-13      1.752899e-03        5.842996e-07
   6   1.003907e-13      1.003907e-01        3.346358e-05
   7  -9.655131e-18     -9.655131e-04       -3.218377e-07
   8  -3.661383e-18     -3.661383e-02       -1.220461e-05
   9   4.813642e-22      4.813642e-04        1.604547e-07
  10   1.353786e-22      1.353786e-02        4.512621e-06
  11  -2.249360e-26     -2.249360e-04       -7.497867e-08
  12  -5.035262e-27     -5.035262e-03       -1.678421e-06
  13   1.007476e-30      1.007476e-04        3.358254e-08
  14   1.88

In [4]:
%reload_ext autoreload
%autoreload 2

### This notebook is for checking the fluctuation theorems in the context ###
### of selection only (no mutations) in finite populations                ###
### Series expansion for fixation probabilites                            ###

### This version does a scan over D equally spaced values of s_alpha, alpha = 1 .. M-1 ###

from functions_expansion import *
import time

start_time = time.time()

folder = "data_fix_prob"
#folder = "./"

##########################

N = 100 # total number of individuals in the population
M = 3 # total number of genotypes

D = 10 # total number of points per fitness range

READX0 = False # reads initial freqs from a file if True

EPS_CONV = 1e-02 # convergence tolerance
NF = 0 # number of unconverged runs

##################################

# Initial population frequencies:
assert M > 1 and type(M) is int, "Illegal M!"

file_x0 = os.path.join(folder, f"fix_probs_expansion.N{N}_M{M}_D{D}.x0.txt")
####
if READX0 == True:
    init_freqs = np.loadtxt(file_x0)
    assert init_freqs.size == M, "Dimension mismatch in initial freqs!"
else:
    init_freqs = np.ones(M) / M # uniform init_freqs
    #init_freqs = [0.2, 0.8] # non-uniform freqs, M=2
    #init_freqs = [0.15, 0.45, 0.40] # non-uniform freqs, M=3
    np.savetxt(file_x0, init_freqs, fmt='%.8f')
    

print(f"M = {M}")
np.set_printoptions(formatter={'float': '{:0.3f}'.format})
print("Initial population frequencies:\n",init_freqs)
np.set_printoptions()


### Set up selection coeffs: ###
eps = 1e-05 # s = 0.0 does not work in Kimura's formula
SMAX = 0.02
sel_coeff = np.linspace(0.0, SMAX, num=D) + eps
sel_shape = tuple(np.repeat(D,M-1)) # shape of the matrix of selection coeff. 

######
fix_probs = np.zeros(sel_shape + (M,)) # global fixation probs

#debug
#print("shape of fix_probs =",fix_probs.shape)

print(f"Starting {D**(M-1)} series expansion(s) ..")
Mstep = int(D**(M-1) // 10) # intervals at which progress is reported

######
sel_vec = np.zeros(M)

for k in range(D**(M-1)): # "global" index over selection coeffs
    
    ####
    if (Mstep > 0 and (k+1)%Mstep == 0) or (Mstep == 0):
        if k+1 == 1:
            print("{} run ..".format(k+1))
        else:
            print("{} runs ..".format(k+1))
            
    
    crd_cur = tuple(np.unravel_index(k, sel_shape))
    assert len(crd_cur) == M-1, "Dimension mismatch!"
    
    #debug
    #print(f"crd_cur = {crd_cur}, type ={type(crd_cur)}")
    
    ### Selection coefficients for each genotype: ###
    for p in range(M-1):
        sel_vec[p] = sel_coeff[crd_cur[p]]
    sel_vec[-1] = 0.0 # s_M = 0.0 by construction
    
    #debug
    #print("sel_vec =",sel_vec)
    
    ####################
    # Build fixation probs given a vector of selection coeffs and initial population freq:
    P_sum = 0.0
    for gamma in range(M-1):
        P_gamma_ds = fixation_probability(x=init_freqs, s=sel_vec, gamma=gamma, N=N, \
                                          tol=-1e-04, max_k=36, return_details=True)
        ######
        crd_cur_ext = crd_cur + (gamma,)
        fix_probs[crd_cur_ext] = P_gamma_ds["P_fix"]
        P_sum += P_gamma_ds["P_fix"]
        
        # Convergence check:
        Pcontr = np.array([term["P_contribution"] for term in P_gamma_ds["terms"]])
        last_ave = np.mean(np.abs(Pcontr[-3:]))
        
        if last_ave > EPS_CONV: # failed to converge
            NF += 1
        
        # Diagnostics:
        #Pcontr = []
        #for term in P_gamma_ds["terms"]:
        #    Pcontr.append(term['P_contribution'])
        
        #if len(Pcontr) < 3:
        #    last_ave = np.mean(np.abs(Pcontr))
        #else:
        #    last_ave = np.mean(np.abs(Pcontr[-3:]))
        
        #if abs(last_ave) > 1e-07:
        #    print(f"WARNING: convergenve to {last_ave} for {sel_vec}!")
        
        #debug
        #print(Pcontr)
    
    crd_cur_ext = crd_cur + (M-1,)
    fix_probs[crd_cur_ext] = max(0.0, 1.0 - P_sum) # last Pfix by normalization
    #fix_probs[crd_cur] = counts/count_sum

    #debug
    #print("fix_prob =",fix_probs[crd_cur])
    
filename = os.path.join(folder, f"fix_probs_expansion.N{N}_M{M}_D{D}.npy")
np.save(filename, fix_probs)

print(f"Finished {D**(M-1)} series expansion(s) ..\n")
if NF == 0:
    print(f"All runs converged at {EPS_CONV} tolerance ..\n")
else:
    print(f"WARNING: {NF} runs did not converge at {EPS_CONV} tolerance ..\n")

# Save any newly generated symbolic data
save_symbolic_caches() # save any changes with default filename

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")


M = 3
Initial population frequencies:
 [0.333 0.333 0.333]
Starting 100 series expansion(s) ..
10 runs ..
20 runs ..
30 runs ..
40 runs ..
50 runs ..
60 runs ..
70 runs ..
80 runs ..
90 runs ..
100 runs ..
Finished 100 series expansion(s) ..

All runs converged at 0.01 tolerance ..

Symbolic caches unchanged; skipping save.

Code ran in 37.9787 seconds = 0.6330 minutes


In [7]:
%reload_ext autoreload
%autoreload 2

### This notebook is for checking the fluctuation theorems in the context ###
### of selection only (no mutations) in finite populations                ###
### Series expansion for fixation probabilites                            ###

### This version uses D sets of [s_1 .. s_M-1] loaded from a file, NO scan is performed ###

from functions_expansion import *
import time

start_time = time.time()

folder = "data_fix_prob"
#folder = "./"

##########################

N = 100 # total number of individuals in the population

#M = 5 # total number of genotypes
#D = 20 # total number of randomly sampled selection coefficient sets
M = 10 # total number of genotypes
D = 10 # total number of randomly sampled selection coefficient sets
#M = 20 # total number of genotypes
#D = 5 # total number of randomly sampled selection coefficient sets

READX0 = False # reads initial freqs from a file if True
READSEL = True # reads selection coefficients from a file if True

EPS_CONV = 1e-02 # convergence tolerance
NF = 0 # number of unconverged runs

# Create a generator instance:
rng = np.random.default_rng()

##################################

# Initial population frequencies:
assert M > 1 and type(M) is int, "Illegal M!"

file_x0 = os.path.join(folder, f"fix_probs_expansion_rsets.N{N}_M{M}_D{D}.x0.txt")
####
if READX0 == True:
    init_freqs = np.loadtxt(file_x0)
    assert init_freqs.size == M, "Dimension mismatch in initial freqs!"
else:
    init_freqs = np.ones(M) / M # uniform init_freqs
    #init_freqs = [0.2, 0.8] # non-uniform freqs, M=2
    #init_freqs = [0.15, 0.45, 0.40] # non-uniform freqs, M=3
    np.savetxt(file_x0, init_freqs, fmt='%.8f')


print(f"M = {M}")
np.set_printoptions(formatter={'float': '{:0.3f}'.format})
print("Initial population frequencies:\n",init_freqs)
np.set_printoptions()


### Set up selection coeffs: ###
filename1 = os.path.join(folder, f"sel_coeff_expansion_rsets.N{N}_M{M}_D{D}.npy")
####
if READSEL == True:
    sel_coeff_set = np.load(filename1)
    assert sel_coeff_set.shape[0] == D and sel_coeff_set.shape[1] == M-1, "Dimension mismatch in sel_coeff_set!"
else:
    eps = 1e-05 # s = 0.0 does not work in Kimura's formula
    SMAX = 0.02
    sel_coeff_set = rng.uniform(0.0, SMAX, size=(D,M-1)) + eps
    np.save(filename1, sel_coeff_set)
    
sel_coeff = np.column_stack((sel_coeff_set, np.zeros(sel_coeff_set.shape[0]))) # add a column of 0 at the end

### Set up selection coeffs: ###
#filename_sel = f"sel_coeff_rsets.N{N}_M{M}_D{D}.npy"
#sel_coeff_orig = np.load(filename_sel)

#sel_coeff = np.column_stack((sel_coeff_orig, np.zeros(sel_coeff_orig.shape[0]))) # add a column of 0 at the end

######
fix_probs = np.zeros((D,M)) # global fixation probs

#debug
#print("shape of fix_probs =",fix_probs.shape)

print(f"Starting {D} series expansion(s) ..")
Mstep = int(D // 5) # intervals at which progress is reported

######
sel_vec = np.zeros(M)

for k in range(D): # "global" index over selection coeffs
    
    ####
    if (Mstep > 0 and (k+1)%Mstep == 0) or (Mstep == 0):
        if k+1 == 1:
            print("{} run ..".format(k+1))
        else:
            print("{} runs ..".format(k+1))
            
    ####################
    # Build fixation probs given a vector of selection coeffs and initial population freq:
    P_sum = 0.0
    for gamma in range(M-1):
        P_gamma_ds = fixation_probability(x=init_freqs, s=sel_coeff[k], gamma=gamma, N=N, \
                                          tol=-1e-04, max_k=36, return_details=True)
        ######
        fix_probs[k,gamma] = P_gamma_ds["P_fix"]
        P_sum += P_gamma_ds["P_fix"]
        
        # Convergence check:
        Pcontr = np.array([term["P_contribution"] for term in P_gamma_ds["terms"]])
        last_ave = np.mean(np.abs(Pcontr[-3:]))
        
        if last_ave > EPS_CONV: # failed to converge
            NF += 1
        
        # Diagnostics:
        #Pcontr = []
        #for term in P_gamma_ds["terms"]:
        #    Pcontr.append(term['P_contribution'])
        
        #if len(Pcontr) < 3:
        #    last_ave = np.mean(np.abs(Pcontr))
        #else:
        #    last_ave = np.mean(np.abs(Pcontr[-3:]))
        
        #if abs(last_ave) > 1e-07:
        #    print(f"WARNING: convergenve to {last_ave} for {sel_vec}!")
        
        #debug
        #print(Pcontr)
    
    fix_probs[k,M-1] = max(0.0, 1.0 - P_sum) # last Pfix by normalization

    #debug
    #print("fix_prob =",fix_probs[crd_cur])


filename2 = os.path.join(folder, f"fix_probs_expansion_rsets.N{N}_M{M}_D{D}.npy")
np.save(filename2, fix_probs)

print(f"Finished {D} series expansion(s) ..\n")
if NF == 0:
    print(f"All runs converged at {EPS_CONV} tolerance ..\n")
else:
    print(f"WARNING: {NF} runs did not converge at {EPS_CONV} tolerance ..\n")

# Save any newly generated symbolic data
save_symbolic_caches() # save any changes with default filename

end_time = time.time()
print(f"\nCode ran in {end_time - start_time:.4f} seconds = {(end_time - start_time)/60.0:.4f} minutes")


M = 10
Initial population frequencies:
 [0.100 0.100 0.100 0.100 0.100 0.100 0.100 0.100 0.100 0.100]
Starting 10 series expansion(s) ..
2 runs ..
4 runs ..
6 runs ..
8 runs ..
10 runs ..
Finished 10 series expansion(s) ..

All runs converged at 0.01 tolerance ..

Symbolic caches unchanged; skipping save.

Code ran in 16.7593 seconds = 0.2793 minutes


In [9]:
%reload_ext autoreload
%autoreload 2

### This notebook is for computing fixation probabilities of a beneficial ###
### mutation on the bkgr of interfering mutations.                        ###
### Series expansion for fixation probabilites                            ###

### This version computes scans over M, SBEN and/or [SMIN,SMAX] ###

from functions_expansion import *
import time

start_time = time.time()

folder = "data_fix_prob"
#folder = "./"

EPS_CONV = 1e-02 # convergence tolerance
NF = 0 # number of unconverged runs

########################## High-res scan over M and SMAX for a few values of SBEN

"""
N = 100 # total number of individuals in the population
D = 50 # total number of randomly sampled selection coefficient sets

N_M = 20 # total number of M values
N_SMIN = 1 # total number of SMIN sets
N_SMAX = 20 # total number of SMAX sets
N_SBEN = 2

Min = 5 # min M
Max = 50 # max M
Msets = np.linspace(Min, Max, num=N_M, dtype=int) # array of M

eps = 1e-08 # small offset (s = 0.0 does not work in Kimura's formula)

SMIN_1 = 0.0
SMIN_2 = 0.0
SMINsets = np.linspace(SMIN_1, SMIN_2, num=N_SMIN) # array of SMIN

SMAX_1 = 0.0
SMAX_2 = 0.02
SMAXsets = np.linspace(SMAX_1, SMAX_2, num=N_SMAX) # array of SMAX

#SBEN = 0.02 # sel_coeff for a pre-specified beneficial mutation

# Beneficial mutation selection coefficient scan
SBEN_1 = 0.0025
SBEN_2 = 0.005
SBENsets = np.linspace(SBEN_1, SBEN_2, num=N_SBEN)
"""

########################## Medium-res scan over M, SMAX and SBEN

BENmut = False # BEN or DEL scenarios

N = 100 # total number of individuals in the population
D = 100 # total number of randomly sampled selection coefficient sets

N_M = 5 # total number of M values
N_SMIN = 1 # total number of SMIN sets
N_SMAX = 5 # total number of SMAX sets
N_SBEN = 5

Min = 5 # min M
Max = 25 # max M
Msets = np.linspace(Min, Max, num=N_M, dtype=int) # array of M

eps = 1e-08 # small offset (s = 0.0 does not work in Kimura's formula)

if BENmut == True:
    SMIN_1 = 0.0 # beneficial interfering mutations
    SMIN_2 = 0.0
else:
    SMIN_1 = -0.01 # deleterious interfering mutations
    SMIN_2 = -0.01
######
SMINsets = np.linspace(SMIN_1, SMIN_2, num=N_SMIN) # array of SMIN

if BENmut == True:
    SMAX_1 = 0.0 # beneficial interfering mutations
    SMAX_2 = 0.02
else:
    SMAX_1 = -0.01 # deleterious interfering mutations
    SMAX_2 = 0.01
######
SMAXsets = np.linspace(SMAX_1, SMAX_2, num=N_SMAX) # array of SMAX

#SBEN = 0.02 # sel_coeff for a pre-specified beneficial mutation

# Beneficial mutation selection coefficient scan
SBEN_1 = 0.0
if BENmut == True:
    SBEN_2 = 0.02 # beneficial interfering mutations
else:
    SBEN_2 = 0.012 # deleterious interfering mutations
######
SBENsets = np.linspace(SBEN_1, SBEN_2, num=N_SBEN)

#########################################################################

# Padding length for selection coefficient arrays
M_pad = int(np.max(Msets))

# Create a generator instance
rng = np.random.default_rng()

# Axis structure:
# sel_coeffs_g[p, b, j, k, l, i]
#   p = selection coefficient index, padded to M_pad
#   b = SBEN index
#   j = M index
#   k = SMIN index
#   l = SMAX index
#   i = D realization index
sel_coeffs_g = np.full(
    (M_pad, N_SBEN, N_M, N_SMIN, N_SMAX, D),
    -1.0,
    dtype=float
)

# fix_probs[b, j, k, l, i]
fix_probs = np.zeros(
    (N_SBEN, N_M, N_SMIN, N_SMAX, D),
    dtype=float
)

cnt = 0
total_steps = D * N_SBEN * N_M * N_SMIN * N_SMAX
Mstep = max(1, total_steps // 10)

for j in range(N_M):
    M = Msets[j]
    
    # uniform freqs:
    #init_freqs = np.ones(M) / M
    
    # single benef. mutant:
    init_freqs = np.full(M, (1 - 1/N)/(M - 1))
    init_freqs[0] = 1/N
    
    assert np.isclose(np.sum(init_freqs), 1.0)

    for i in range(D):
        for k in range(N_SMIN):
            for l in range(N_SMAX):

                # Check that the random interval is valid
                smin = SMINsets[k]
                smax = SMAXsets[l]
                if smin > smax:
                    raise ValueError(
                        f"Invalid interval: SMIN={smin} > SMAX={smax}"
                    )
                
                # variable var, fixed mean frequency setup:
                #init_freqs = np.full(M, (1 - smax)*(1 - 1/N)/(M - 2))
                #init_freqs[0] = 1/N
                #init_freqs[1] = (1 - init_freqs[0])*smax # smax=q here
                
                #assert np.isclose(np.sum(init_freqs), 1.0)
                
                
                # ------------------------------------------------------------
                # Draw the background selection coefficients ONCE.
                #
                # Full selection coefficient vector has length M:
                #   index 0        : beneficial mutation, SBEN
                #   indices 1:M-1  : random background coefficients
                #   index M-1      : terminal/reference type with s = 0.0
                #
                # Therefore the genuinely random background entries are
                # indices 1,...,M-2, i.e. M-2 numbers.
                # ------------------------------------------------------------
                bg_random = rng.uniform(
                    smin,
                    smax,
                    size=M-2
                ) + eps
                
                
                for b in range(N_SBEN):
                    SBEN = SBENsets[b]

                    #if (cnt + 1) % Mstep == 0 or cnt + 1 == 1:
                    #    print(f"{cnt + 1} runs ..")
                    #cnt += 1
                    
                    ####
                    if (Mstep > 0 and (cnt+1)%Mstep == 0) or (Mstep == 0):
                        if cnt+1 == 1:
                            print("{} run ..".format(cnt+1))
                        else:
                            print("{} runs ..".format(cnt+1))
                    cnt += 1

                    # Build full selection coefficient array
                    sel_coeff_set = np.empty(M, dtype=float)
                    sel_coeff_set[0] = SBEN
                    sel_coeff_set[1:M-1] = bg_random
                    sel_coeff_set[M-1] = 0.0
                    
                    #sel_coeff_set = np.zeros(M)
                    #sel_coeff_set[0] = SBEN + smin * smax/(1-smax)
                    #sel_coeff_set[1] = smin/(1-smax)

                    # Safety checks
                    assert sel_coeff_set.size == M
                    assert np.isclose(sel_coeff_set[0], SBEN)
                    assert np.isclose(sel_coeff_set[-1], 0.0)
                    
                    # Store valid entries; padding stays -1.0
                    sel_coeffs_g[:M, b, j, k, l, i] = sel_coeff_set

                    # Compute fixation probability of gamma = 0
                    P_gamma_ds = fixation_probability(
                        x=init_freqs,
                        s=sel_coeff_set,
                        gamma=0,
                        N=N,
                        tol=-1e-04,
                        max_k=36,
                        return_details=True
                    )
                    
                    # Convergence check:
                    Pcontr = np.array([term["P_contribution"] for term in P_gamma_ds["terms"]])
                    last_ave = np.mean(np.abs(Pcontr[-3:]))

                    if last_ave > EPS_CONV: # failed to converge
                        NF += 1

                    fix_probs[b, j, k, l, i] = P_gamma_ds["P_fix"]

                    
filename1 = os.path.join(
    folder,
    f"sel_coeff_expan_rsets.SBEN_{SBEN_1}-{SBEN_2}_NSBEN{N_SBEN}_"
    f"N{N}_M_{Min}-{Max}_SMIN_{SMIN_1}-{SMIN_2}_"
    f"SMAX_{SMAX_1}-{SMAX_2}_D{D}.npy"
)

filename2 = os.path.join(
    folder,
    f"fprobs_expan_rsets.SBEN_{SBEN_1}-{SBEN_2}_NSBEN{N_SBEN}_"
    f"N{N}_M_{Min}-{Max}_SMIN_{SMIN_1}-{SMIN_2}_"
    f"SMAX_{SMAX_1}-{SMAX_2}_D{D}.npy"
)

np.save(filename1, sel_coeffs_g)
np.save(filename2, fix_probs)

print(f"\nFinished {total_steps} series expansion(s) ..\n")
if NF == 0:
    print(f"All runs converged at {EPS_CONV} tolerance ..\n")
else:
    print(f"WARNING: {NF} runs did not converge at {EPS_CONV} tolerance ..\n")

# Save any newly generated symbolic data
save_symbolic_caches() # save any changes with default filename

end_time = time.time()

print(
    f"\nCode ran in {end_time - start_time:.4f} seconds "
    f"= {(end_time - start_time)/60.0:.4f} minutes"
)



1250 runs ..
2500 runs ..
3750 runs ..
5000 runs ..
6250 runs ..
7500 runs ..
8750 runs ..
10000 runs ..
11250 runs ..
12500 runs ..

Finished 12500 series expansion(s) ..

All runs converged at 0.01 tolerance ..

Symbolic caches unchanged; skipping save.

Code ran in 2237.9320 seconds = 37.2989 minutes


In [18]:
%reload_ext autoreload
%autoreload 2

### This notebook is for computing fixation probabilities of a beneficial ###
### mutation on the bkgr of interfering mutations.                        ###
### Series expansion for fixation probabilites                            ###

### This version computes the q-scan with a fixed mean of beneficial and deleterious mutations ###
### It also outputs Kimura's fixation probabilities for a side-by-side comparison ###

from functions_expansion import *
import numpy as np
import os
import time

# ============================================================
# 0. Parameters
# ============================================================

start_time = time.time()

folder = "data_fix_prob"
os.makedirs(folder, exist_ok=True)

EPS_CONV = 1e-02 # convergence tolerance
NF = 0 # number of unconverged runs

N = 100

# Grid sizes
N_SB = 11
#N_Q  = 11 # with q_eps = 1e-3
N_Q  = 9 # with q_eps = 0.0
N_Sh = 11

# Parameter ranges
SB_1 = 0.0
SB_2 = 0.02
SBsets = np.linspace(SB_1, SB_2, num=N_SB)

Sh_1 = 0.0
Sh_2 = 0.02
Shsets = np.linspace(Sh_1, Sh_2, num=N_Sh)

# q must avoid 0 and 1 [actually, only 1]
#q_eps = 1e-3
q_eps = 0.0
q_max = 0.8
#qsets = np.linspace(q_eps, q_max - q_eps, num=N_Q)
qsets = np.linspace(q_eps, q_max, num=N_Q)

# M values
#Msets = np.arange(5, 26, 5)
Msets = np.arange(3, 16, 3)
N_M = len(Msets)

# Initial frequency of the new mutant
x0 = 1.0 / N

# Expansion parameters
tol = -1e-04
max_k = 36
s_max_poss = 0.025 # max possible sel_coeff above which the expansion may fail

# ============================================================
# 1. Output arrays
# ============================================================

# Axis order:
#   axis 0: M
#   axis 1: S_B
#   axis 2: q
#   axis 3: S_h

fix_probs_expansion = np.full((N_M, N_SB, N_Q, N_Sh), np.nan)
fix_probs_kimura    = np.full((N_M, N_SB, N_Q, N_Sh), np.nan)

valid_expansion = np.zeros((N_M, N_SB, N_Q, N_Sh), dtype=bool)

# Optional: store full x and s vectors, padded to max M
M_pad = int(np.max(Msets))

x_g = np.full((M_pad, N_M, N_Q), np.nan)
s_g = np.full((M_pad, N_M, N_SB, N_Q, N_Sh), np.nan)

# ============================================================
# 2. Numerically stable Kimura formula
# ============================================================

def kimura_fixation_probability_bare_SB(N, SB, x0, atol=1e-14):
    """
    Kimura fixation probability with bare S_B:

        P_fix = (1 - exp(-2 N S_B x0)) / (1 - exp(-2 N S_B))

    Uses expm1 and handles the neutral limit S_B -> 0.
    """
    SB = np.asarray(SB, dtype=float)
    x0 = np.asarray(x0, dtype=float)

    SB, x0 = np.broadcast_arrays(SB, x0)

    neutral = np.isclose(SB, 0.0, atol=atol)

    numerator = -np.expm1(-2.0 * N * SB * x0)
    denominator = -np.expm1(-2.0 * N * SB)

    out = np.array(x0, dtype=float, copy=True)

    np.divide(
        numerator,
        denominator,
        out=out,
        where=~neutral
    )

    if out.ndim == 0:
        return out.item()

    return out

# ============================================================
# 3. Main scan
# ============================================================

cnt = 0
total_steps = N_M * N_SB * N_Q * N_Sh
report_step = max(1, total_steps // 10)

for j, M in enumerate(Msets):

    for iq, q in enumerate(qsets):

        # ------------------------------------------------------------
        # Initial frequency vector
        #
        # x = (
        #     x0,
        #     (1 - x0) q,
        #     (1 - x0)(1 - q)/(M - 2),
        #     ...,
        #     (1 - x0)(1 - q)/(M - 2)
        # )
        # ------------------------------------------------------------
        x = np.empty(M, dtype=float)

        x[0] = x0
        x[1] = (1.0 - x0) * q
        x[2:] = (1.0 - x0) * (1.0 - q) / (M - 2)

        assert np.isclose(np.sum(x), 1.0), "Initial frequencies do not sum to 1"
        assert np.all(x >= 0.0), "Negative initial frequency found"

        # Store padded x vector
        x_g[:M, j, iq] = x

        for iSB, SB in enumerate(SBsets):
            for iSh, Sh in enumerate(Shsets):

                cnt += 1

                if cnt == 1 or cnt % report_step == 0:
                    print(f"{cnt} / {total_steps} runs ..")

                # ------------------------------------------------------------
                # Selection coefficient vector
                #
                # s = (
                #     S_B + S_h q/(1-q),
                #     S_h/(1-q),
                #     0,
                #     ...,
                #     0
                # )
                # ------------------------------------------------------------
                s = np.zeros(M, dtype=float)

                s[0] = SB + Sh * q / (1.0 - q)
                s[1] = Sh / (1.0 - q)
                s[2:] = 0.0

                # Last selection coefficient must be exactly 0.0
                assert np.isclose(s[-1], 0.0), "Last selection coefficient is not 0.0"

                # Store padded s vector
                s_g[:M, j, iSB, iq, iSh] = s
                
                max_s = np.max(s)
                
                if max_s > s_max_poss:
                    
                    fix_probs_expansion[j, iSB, iq, iSh] = np.nan
                    valid_expansion[j, iSB, iq, iSh] = False
                
                else:

                    # ------------------------------------------------------------
                    # Exact / expansion fixation probability of mutant i = 0
                    # ------------------------------------------------------------
                    P_gamma_ds = fixation_probability(
                        x=x,
                        s=s,
                        gamma=0,
                        N=N,
                        tol=tol,
                        max_k=max_k,
                        return_details=True
                    )
                    
                    # Convergence check:
                    Pcontr = np.array([term["P_contribution"] for term in P_gamma_ds["terms"]])
                    last_ave = np.mean(np.abs(Pcontr[-3:]))
                    
                    if last_ave > EPS_CONV: # failed to converge
                        NF += 1
                    
                    fix_probs_expansion[j, iSB, iq, iSh] = P_gamma_ds["P_fix"]
                    valid_expansion[j, iSB, iq, iSh] = True

                # ------------------------------------------------------------
                # Kimura comparison using bare S_B only
                # ------------------------------------------------------------
                fix_probs_kimura[j, iSB, iq, iSh] = kimura_fixation_probability_bare_SB(
                    N=N,
                    SB=SB,
                    x0=x0
                )

print(f"\nFinished {total_steps} fixation-probability calculations.\n")

n_valid = np.sum(valid_expansion)
n_total = valid_expansion.size
n_skipped = n_total - n_valid

print(f"\nValid expansion points (runs):   {n_valid} / {n_total}")
print(f"Skipped expansion points (runs): {n_skipped} / {n_total}")
print(f"Skipped fraction:         {n_skipped / n_total:.4f}\n")

if NF == 0:
    print(f"All valid runs converged at {EPS_CONV} tolerance ..\n")
else:
    print(f"WARNING: {NF} valid runs did not converge at {EPS_CONV} tolerance ..\n")

# ============================================================
# 4. Save results
# ============================================================

out_file = os.path.join(
    folder,
    f"interference_model_scan_"
    f"N{N}_M_{Msets[0]}-{Msets[-1]}_"
    f"SB_{SB_1}-{SB_2}_NSB{N_SB}_"
    f"q_{qsets[0]:.4f}-{qsets[-1]:.4f}_NQ{N_Q}_"
    f"Sh_{Sh_1}-{Sh_2}_NSh{N_Sh}.npz"
)

np.savez_compressed(
    out_file,
    fix_probs_expansion=fix_probs_expansion,
    fix_probs_kimura=fix_probs_kimura,
    valid_expansion=valid_expansion,
    x_g=x_g,
    s_g=s_g,
    N=N,
    x0=x0,
    Msets=Msets,
    SBsets=SBsets,
    qsets=qsets,
    Shsets=Shsets,
    tol=tol,
    max_k=max_k,
    s_max_poss=s_max_poss
)

print("Saved:", out_file, "\n")

# Save symbolic caches if your expansion code generated new symbolic data
save_symbolic_caches()

end_time = time.time()
print(
    f"\nCode ran in {end_time - start_time:.4f} seconds "
    f"= {(end_time - start_time)/60.0:.4f} minutes\n"
)


1 / 5445 runs ..
544 / 5445 runs ..
1088 / 5445 runs ..
1632 / 5445 runs ..
2176 / 5445 runs ..
2720 / 5445 runs ..
3264 / 5445 runs ..
3808 / 5445 runs ..
4352 / 5445 runs ..
4896 / 5445 runs ..
5440 / 5445 runs ..

Finished 5445 fixation-probability calculations.


Valid expansion points (runs):   3575 / 5445
Skipped expansion points (runs): 1870 / 5445
Skipped fraction:         0.3434

All valid runs converged at 0.01 tolerance ..

Saved: data_fix_prob/interference_model_scan_N100_M_3-15_SB_0.0-0.02_NSB11_q_0.0000-0.8000_NQ9_Sh_0.0-0.02_NSh11.npz 

Symbolic caches unchanged; skipping save.

Code ran in 636.6681 seconds = 10.6111 minutes



In [19]:
#print(SBsets)
#print(Shsets)

np.linspace(0.0, 0.02, num=10)

array([0.        , 0.00222222, 0.00444444, 0.00666667, 0.00888889,
       0.01111111, 0.01333333, 0.01555556, 0.01777778, 0.02      ])

In [25]:
print(np.linspace(0.0, 0.012, num=5))
print(np.linspace(-0.01, 0.01, num=5))

[0.    0.003 0.006 0.009 0.012]
[-0.01  -0.005  0.     0.005  0.01 ]


In [17]:
q_eps = 0.0 # 1e-03
q_max = 0.8
qsets = np.linspace(q_eps, q_max, num=9)

print(qsets)

[0.  0.1 0.2 0.3 0.4 0.5 0.6 0.7 0.8]
